# Statistical fine-mapping

> **Important notes for this notebook**
>
> Finding a genome-wide significant locus in a GWAS (`GWAS5` or `GWAS9`)
> is only the first step. Nearby SNPs are correlated through **linkage
> disequilibrium (LD)**, so one causal variant makes many non-causal
> neighbours significant too. If a locus contains several causal
> variants, the SNP with the smallest $p$-value may not be causal at
> all. In this notebook, we use **statistical fine-mapping** with
> **SuSiE** (*Sum of Single Effects*; [Wang et al.,
> 2020](https://doi.org/10.1111/rssb.12388); [Zou et al.,
> 2022](https://doi.org/10.1371/journal.pgen.1010299)) to find the
> likely causal variants and summarize them as **95% credible sets**.
>
> ## Learning outcomes
>
> - **Explain** why single-SNP GWAS tests cannot tell causal variants
>   apart from SNPs in LD with them
> - **Understand** the **SuSiE** model and how it differs from earlier
>   fine-mapping methods (`FINEMAP`, `CAVIAR`)
> - **Interpret** **posterior inclusion probabilities (PIPs)** and **95%
>   credible sets**
> - **Run** fine-mapping in `R` with individual-level data (`susie()`)
>   and with GWAS summary statistics (`susie_rss()`)
> - **Detect** a mismatch between the LD reference panel and the GWAS
>   sample
>
> ## How to make this notebook work
>
> - In this notebook, we will use both `bash command line` and `R`
>   programming languages. Remember to **change the kernel** whenever
>   you transition from one language to the other
>   (`Kernel --> Change Kernel`) indicated by the languages’ images. We
>   will first run `Bash` commands.
>
> <img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash
> kernel

> **Some theory**
>
> ### Intuition First: Why isn’t the top GWAS SNP always the causal variant?
>
> - **SNPs in LD look alike**: If one SNP is causal, every SNP in strong
>   LD with it ($r^2 \approx 0.95$) gets an almost equally strong
>   association when SNPs are tested one at a time. The GWAS cannot tell
>   which of them is causal.
> - **Two causal SNPs can create a misleading peak**: If a locus has two
>   causal SNPs, a non-causal SNP that is partly correlated with both
>   (say $r \approx 0.6$ with each) can pick up signal from both and end
>   up with a **smaller $p$-value than either causal SNP**. We call it a
>   “sponge” SNP (the technical term is *synthetic association*).
>
> **Fine-mapping** solves this by fitting all SNPs in the locus
> **together** in one regression model and asking: *which small set of
> SNPs explains the whole association signal?*
>
> ### Why Single-SNP GWAS Cannot Pinpoint Causal Variants
>
> Suppose a locus has $p$ SNPs and the true model is
>
> $$y = X b + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$
>
> where $X$ holds the standardized genotypes of the $N$ individuals
> ($N \times p$) and $b = (b_1, \dots, b_p)$ contains the true effects,
> most of them zero. The single-SNP GWAS $Z$-scores
> $z = (z_1, \dots, z_p)$ then satisfy approximately
>
> $$\mathbb{E}[z] \approx \sqrt{N}\, R\, b, \quad \text{Cov}(z) = R$$
>
> where $R = \frac{1}{N} X^\top X$ is the $p \times p$ **LD
> (correlation) matrix** of the locus. Because $b$ is multiplied by
> $R$: 1. **LD tagging**: A non-causal SNP $k$ ($b_k = 0$) that is
> correlated with a causal SNP $j$ gets
> $\mathbb{E}[z_k] \approx \sqrt{N}\, r_{jk}\, b_j$, which is not zero.
> 2. **Synthetic association**: A SNP correlated with two causal SNPs
> adds up both signals, and can get the largest $Z$-score in the locus.
>
> ### Statistical Fine-Mapping: PIPs and 95% Credible Sets
>
> Fine-mapping fits all $p$ SNPs together in a Bayesian regression that
> assumes only a few SNPs are causal. It reports two things:
>
> - **Posterior inclusion probability (PIP)**: the probability, given
>   the data, that SNP $j$ is causal:
>   $$\text{PIP}_j = \Pr(b_j \neq 0 \mid \text{data})$$
> - **95% credible set (CS)**: a small group of SNPs that contains a
>   causal variant with at least 95% probability.
>   - *Why we need credible sets as well as PIPs — the “locked rooms”
>     analogy*: a prize (the causal variant) is behind one of **5
>     identical locked rooms** (5 SNPs in near-perfect LD,
>     $r^2 \approx 0.99$). The data cannot tell the rooms apart, so each
>     SNP gets $\text{PIP} \approx 0.20$. If we only looked for SNPs
>     with $\text{PIP} > 0.9$, we would miss the signal. The credible
>     set groups the 5 rooms ($\sum \text{PIP} \ge 0.95$) and says:
>     *“one of these 5 SNPs is causal, and the other SNPs in the locus
>     can be ruled out.”*
>
> ### How `SuSiE` (Sum of Single Effects) Works
>
> Older methods (`CAVIAR`, `FINEMAP`) search over combinations of causal
> SNPs, which becomes slow when there are several causal variants.
> **SuSiE** ([Wang et al., 2020](https://doi.org/10.1111/rssb.12388))
> instead writes the effects as a **sum of $L$ single effects**:
>
> $$b = \sum_{l=1}^{L} b^{(l)}$$
>
> where each single effect $b^{(l)}$ has **exactly one** non-zero entry:
> one causal SNP (chosen with probabilities
> $\alpha_{l1}, \dots, \alpha_{lp}$) with effect size
> $\beta^{(l)} \sim \mathcal{N}(0, \sigma_{0l}^2)$.
>
> SuSiE fits the effects one at a time, repeatedly (*iterative Bayesian
> stepwise selection*): each single effect is fitted to what the other
> $L - 1$ effects have not yet explained. Each effect $l$ gives its own
> credible set (sort SNPs by $\alpha_{lj}$ and add them until the sum
> reaches 0.95). The PIP of SNP $j$ combines all effects:
>
> $$\text{PIP}_j = 1 - \prod_{l=1}^{L} (1 - \alpha_{lj})$$
>
> | Feature | LD Clumping (`PLINK --clump`) | `FINEMAP` / `CAVIAR` | `SuSiE` / `SuSiE-RSS` (`susieR`) |
> |------------------|------------------|------------------|------------------|
> | **Approach** | Keep the top SNP, remove SNPs in LD with it (rule of thumb) | Search over combinations of causal SNPs | Sum of single effects (Bayesian, fast) |
> | **Gives PIPs?** | No | Yes | **Yes** |
> | **One credible set per signal?** | No | Not directly | **Yes** |
> | **Input** | Summary statistics + LD reference | Summary statistics + LD matrix $R$ | **Individual-level data (`susie`) or summary statistics + $R$ (`susie_rss`)** |
>
> > **Quick Concept Check (from Lecture 10)**: - *SuSiE reports a 95%
> > credible set of 5 SNPs in perfect LD ($r^2 = 1$), each with
> > $\text{PIP} = 0.20$. Is this a false positive?* **No.** We are at
> > least 95% sure that one causal variant is among these 5 SNPs; the
> > data just cannot say which one ($5 \times 0.20 = 1.00$). - *When can
> > the top GWAS SNP get $\text{PIP} \approx 0$?* When it is a
> > non-causal “sponge” SNP that is correlated with two causal SNPs.

# Preparing Locus Data for Fine-Mapping

### Extracting a GWAS Locus and Local LD Matrix ($R$)

In practice, after a genome-wide scan (`GWAS5` or `GWAS9`), we take a
window of about 500 kb–1 Mb around each genome-wide significant peak.

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash
kernel.

Let’s link the `Data` folder, create `Results/GWAS10`, and use `PLINK`
to extract a 200-SNP locus on chromosome 1 from `Data/EUR.QC`. We create
two inputs for fine-mapping: - `--extract <file>`: keeps only the 200
SNPs in our locus. - `--recode A`: writes the genotype matrix
(`locus_geno.raw`), with `0`, `1` or `2` copies of the minor allele per
individual and SNP (used by `susie()`). - `--r square`: computes the
$200 \times 200$ LD correlation matrix $R$ (`locus_ld.ld`, used by
`susie_rss()`).

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS10

# Select a 200-SNP window on Chromosome 1
awk '$1 == 1 {print $2}' Data/EUR.QC.bim | head -n 500 | tail -n 200 > Results/GWAS10/locus.snps

# Save SNP coordinates (SNP, CHR, BP) for locus plotting
awk '$1 == 1 {print $2, $1, $4}' Data/EUR.QC.bim | head -n 500 | tail -n 200 > Results/GWAS10/locus.info

# 1. Export additive genotype matrix (0/1/2 dosages) for individual-level SuSiE
plink \
  --bfile Data/EUR.QC \
  --extract Results/GWAS10/locus.snps \
  --recode A \
  --out Results/GWAS10/locus_geno

# 2. Compute the p x p local LD correlation matrix (R) for summary-statistics SuSiE-RSS
plink \
  --bfile Data/EUR.QC \
  --extract Results/GWAS10/locus.snps \
  --r square \
  --out Results/GWAS10/locus_ld

echo "Extracted locus SNPs: $(wc -l < Results/GWAS10/locus.snps)"

### Simulating a Multi-Causal Locus and Running Marginal GWAS in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

In `R`, let’s load `susieR`, read the genotype matrix $X$ and the LD
matrix $R$, and simulate a phenotype with **two causal SNPs** in this
locus (`causal_idx = c(48, 140)`). Together they explain 12% of the
phenotypic variance. SNP 48 lies in a block of SNPs in strong LD
($r^2 > 0.85$), while SNP 140 has no close proxies. *(Fine-mapping needs
densely genotyped SNPs, so here we use `Data/EUR.QC` rather than the
sparser 50,000-SNP `Data/SIM.QC` cohort from `GWAS8` and `GWAS9`.)*

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 9, repr.plot.height = 7)

if (!requireNamespace("susieR", quietly = TRUE)) {
  install.packages("susieR", repos = "https://cloud.r-project.org")
}
suppressMessages(suppressWarnings(library(susieR)))
suppressMessages(suppressWarnings(library(ggplot2)))

# 1. Load genotype matrix X (N x p), SNP coordinates, and LD matrix R (p x p)
raw_geno   <- read.table("Results/GWAS10/locus_geno.raw", header = TRUE)
locus_info <- read.table("Results/GWAS10/locus.info", header = FALSE,
                         col.names = c("SNP", "CHR", "BP"))
R <- as.matrix(read.table("Results/GWAS10/locus_ld.ld", header = FALSE))

# Columns 7..ncol in PLINK .raw are the SNP dosages
X <- as.matrix(raw_geno[, 7:ncol(raw_geno)])
colnames(X) <- locus_info$SNP

# Mean-impute any sporadic missing genotypes
for (j in seq_len(ncol(X))) {
  X[is.na(X[, j]), j] <- mean(X[, j], na.rm = TRUE)
}

n <- nrow(X)
p <- ncol(X)
cat("Sample size (N):", n, "| Locus SNPs (p):", p, "\n")

In [ ]:
# 2. Simulate a phenotype with 2 true causal variants in the locus
set.seed(2025)
causal_idx <- c(48, 140)
b_true <- rep(0, p)
b_true[causal_idx] <- c(0.45, -0.45)

# Standardize X to mean 0, variance 1 for clean effect simulation
X_std <- scale(X)
g_locus <- as.numeric(X_std %*% b_true)
# Calibrate residual noise so the locus explains ~12% of phenotypic variance
sigma_e <- sqrt(var(g_locus) * (1 - 0.12) / 0.12)
y <- g_locus + rnorm(n, mean = 0, sd = sigma_e)

# 3. Compute marginal single-SNP GWAS summary statistics (as PLINK/LDAK-KVIK would output)
uni_reg <- univariate_regression(X, y)
z_scores <- uni_reg$betahat / uni_reg$sebetahat
p_values <- 2 * pnorm(-abs(z_scores))

cat("True causal SNPs:", locus_info$SNP[causal_idx], "\n")
cat("Top marginal GWAS hit:", locus_info$SNP[which.min(p_values)],
    "(p =", signif(min(p_values), 3), ")\n")

# Individual-Level Fine-Mapping with `susie()`

With individual-level genotypes $X$ and phenotype $y$, we run
`susie(X, y, L = 10)`: - `L = 10` is the **maximum** number of causal
signals in the locus. Effects that are not needed are switched off
automatically (their prior variance is estimated as zero), so `L = 10`
works whether there are 1, 2 or more causal variants. -
`summary(fitted_ind)$cs` lists the **95% credible sets**: - `cs`: the
number of the credible set (`1`, `2`, …) - `cs_log10bf`: the evidence
for this signal, as $\log_{10}(\text{Bayes factor})$ - `cs_avg_r2` /
`cs_min_r2`: the average and minimum LD ($r^2$) between the SNPs in the
set (high values mean a “pure” set of SNPs in strong LD) - `variable`:
the positions (indices) of the SNPs in the set

In [ ]:
# Fit individual-level SuSiE with up to L = 10 single effects
fitted_ind <- susie(X, y, L = 10)

# Print summary of discovered 95% Credible Sets
print(summary(fitted_ind)$cs)

# Check whether our two true causal variants (indices 48 and 140) are captured in the Credible Sets
cs_list <- fitted_ind$sets$cs
for (cs_name in names(cs_list)) {
  snps_in_cs <- cs_list[[cs_name]]
  captured <- intersect(snps_in_cs, causal_idx)
  cat(cs_name, "contains", length(snps_in_cs), "SNP(s):", paste(snps_in_cs, collapse = ", "),
      "| True causal index captured:", ifelse(length(captured) > 0, captured, "None"), "\n")
}

Now let’s plot the **single-SNP GWAS results** (`-log10(p)`) above the
**SuSiE results** (PIPs), with the credible sets and the two true causal
SNPs marked.

In [ ]:
# Assign Credible Set membership for plotting
cs_membership <- rep("None", p)
for (cs_name in names(cs_list)) {
  cs_membership[cs_list[[cs_name]]] <- paste0("Credible Set ", cs_name)
}

plot_df <- data.frame(
  Index    = 1:p,
  BP_Mb    = locus_info$BP / 1e6,
  NegLogP  = -log10(p_values),
  PIP      = fitted_ind$pip,
  CS       = cs_membership,
  IsCausal = ifelse(1:p %in% causal_idx, "True Causal SNP", "Non-Causal SNP")
)

par(mfrow = c(2, 1), mar = c(4, 4.5, 2.5, 1))
# Panel 1: Marginal GWAS -log10(p)
plot(plot_df$BP_Mb, plot_df$NegLogP,
     pch = ifelse(plot_df$IsCausal == "True Causal SNP", 17, 19),
     col = ifelse(plot_df$IsCausal == "True Causal SNP", "tomato", "steelblue"),
     cex = ifelse(plot_df$IsCausal == "True Causal SNP", 1.6, 1.0),
     xlab = "", ylab = expression(-log[10](italic(p))),
     main = "Marginal GWAS Association Signal Across Locus (Confounded by LD)")
abline(h = -log10(0.05 / p), lty = 2, col = "grey40")
legend("topright", legend = c("True Causal SNP", "Tagging / Null SNP"),
       col = c("tomato", "steelblue"), pch = c(17, 19), bty = "n")

# Panel 2: SuSiE Posterior Inclusion Probabilities (PIP) & Credible Sets
susie_plot(fitted_ind, y = "PIP", b = b_true,
           main = "SuSiE Fine-Mapping: Posterior Inclusion Probabilities (PIPs) & 95% Credible Sets")

Compare the two panels: - **Top panel (single-SNP GWAS)**: many SNPs
near the two causal SNPs have small $p$-values because of LD. The top
GWAS hit printed above is **not** one of the causal SNPs, but a SNP in
LD with SNP 48. - **Bottom panel (SuSiE PIPs)**: fitting all SNPs
together reduces the signal to **two 95% credible sets** (colored
circles), each containing one true causal SNP (red). One set contains a
single SNP with PIP close to 1. The other contains 4 SNPs in strong LD
that share the probability between them—the “locked rooms” situation
from the theory section.

# Summary-Statistics Fine-Mapping with `susie_rss()`

Most large GWAS do not share individual-level genotypes. Instead, we
usually have: 1. The GWAS summary statistics: $Z$-scores (`z_scores`),
or $\hat{\beta}$ and $\text{SE}$ 2. The LD matrix $R$ of the locus,
ideally from the GWAS sample itself (**in-sample LD**), otherwise from a
large reference panel of the same ancestry 3. The GWAS sample size `n`

Let’s run **`susie_rss()`** with only `z_scores`, `R` and `n`, and look
at the credible sets.

In [ ]:
# Run SuSiE-RSS using only summary Z-scores, LD matrix R, and sample size n
fitted_rss <- susie_rss(z = z_scores, R = R, n = n, L = 10)

# Inspect the 95% Credible Sets from summary statistics
print(summary(fitted_rss)$cs)

> **Stop - Read - Solve**
>
> - **Q1. Compare the PIPs from individual-level data (`fitted_ind$pip`)
>   with the PIPs from summary statistics and in-sample LD
>   (`fitted_rss$pip`). Make a scatter plot in `ggplot2` and compute
>   their correlation.**
> - **Q2. For each credible set in `fitted_ind$sets$cs`, print (a) the
>   number of SNPs, (b) the largest PIP in the set, and (c) the sum of
>   the PIPs in the set. How can a credible set reach 95% when none of
>   its SNPs has $\text{PIP} > 0.95$?**
> - **Q3. What happens if the LD matrix comes from a small reference
>   panel? Compute `R_small <- cor(X[1:60, ])` from only 60 individuals,
>   check the LD consistency measure $s$ with
>   `estimate_s_rss(z = z_scores, R = R_small, n = n)`, and run
>   `susie_rss(z = z_scores, R = R_small, n = n, L = 10)`. How does the
>   poor LD matrix affect the results?**
>
> Hint: Compare `estimate_s_rss(z = z_scores, R = R, n = n)` (in-sample
> LD, $s \approx 0$) with
> `estimate_s_rss(z = z_scores, R = R_small, n = n)`. A larger $s$ means
> that $z$ and $R$ do not fit together well.

In [ ]:
# Write your code here


> **Solution**
>
> In `R`, we compare `fitted_ind$pip` with `fitted_rss$pip`, look at the
> PIPs within each credible set, and check the LD measure $s$ for the
> small reference panel:
>
> ``` bash
> options(repr.plot.width = 7, repr.plot.height = 4.5)
>
> # Q1. Compare individual-level SuSiE PIPs vs. summary-statistics SuSiE-RSS PIPs
> pip_comp <- data.frame(
>   SNP      = locus_info$SNP,
>   PIP_ind  = fitted_ind$pip,
>   PIP_rss  = fitted_rss$pip,
>   IsCausal = ifelse(1:p %in% causal_idx, "True Causal SNP", "Non-Causal SNP")
> )
>
> cat("Pearson correlation between susie() and susie_rss() PIPs:",
>     round(cor(pip_comp$PIP_ind, pip_comp$PIP_rss), 5), "\n\n")
>
> p_pip <- ggplot(pip_comp, aes(x = PIP_ind, y = PIP_rss, color = IsCausal)) +
>   geom_point(size = 2.5, alpha = 0.85) +
>   geom_abline(intercept = 0, slope = 1, linetype = "dashed") +
>   scale_color_manual(values = c("True Causal SNP" = "tomato", "Non-Causal SNP" = "steelblue")) +
>   theme_bw(base_size = 13) +
>   labs(
>     title = "SuSiE Individual-Level vs. Summary-Statistics (SuSiE-RSS) PIPs",
>     x = "Individual-Level PIP (susie)",
>     y = "Summary-Statistics PIP (susie_rss with in-sample LD)",
>     color = "Variant Class"
>   )
>
> show(p_pip)
>
> # Q2. Inspect PIPs inside each 95% Credible Set
> for (cs_name in names(fitted_ind$sets$cs)) {
>   idx <- fitted_ind$sets$cs[[cs_name]]
>   cat(cs_name, "| Size:", length(idx),
>       "| Max PIP:", round(max(fitted_ind$pip[idx]), 3),
>       "| Sum of PIPs:", round(sum(fitted_ind$pip[idx]), 3), "\n")
> }
>
> # Q3. Diagnose and evaluate noisy/mismatched LD reference matrix
> R_small <- cor(X[1:60, ])
> R_small[is.na(R_small)] <- 0
> diag(R_small) <- 1
>
> s_insample <- estimate_s_rss(z = z_scores, R = R, n = n)
> s_small    <- estimate_s_rss(z = z_scores, R = R_small, n = n)
> cat("\nLD diagnostic s (In-sample R):   ", round(s_insample, 4), "\n")
> cat("LD diagnostic s (Small panel R): ", round(s_small, 4), "\n")
>
> fitted_rss_small <- susie_rss(z = z_scores, R = R_small, n = n, L = 10)
> cat("Number of Credible Sets with noisy LD:", length(fitted_rss_small$sets$cs), "\n")
> ```
>
> - **Q1**: With the exact **in-sample LD matrix**, `susie_rss()` gives
>   practically the same PIPs as `susie()` on the full genotypes
>   (correlation $\approx 0.999$). The credible set for the stronger
>   signal is identical; for the weaker signal, `susie_rss()` adds one
>   extra correlated SNP to the set.
> - **Q2**: When a causal SNP is in strong LD with other SNPs, the
>   probability is shared between them, so no single SNP reaches
>   $\text{PIP} > 0.95$, but their **sum** does. In our run, the 4-SNP
>   credible set has a largest PIP of only about $0.33$, while its PIPs
>   sum to about $1$. Always report the full credible set, not only the
>   top SNP.
> - **Q3**: The 60-person LD matrix is noisy and does not match the GWAS
>   sample, which shows up as a larger $s$ (about $0.57$, versus $0$
>   with in-sample LD). SuSiE-RSS then cannot correctly separate causal
>   SNPs from SNPs in LD with them: in our run, it reported 5 credible
>   sets instead of 2, so some are false positives. Use in-sample LD
>   when possible, or a large reference panel of the same ancestry.

------------------------------------------------------------------------

### Conclusion

In this tutorial, we used `SuSiE` to turn broad GWAS peaks into
**posterior inclusion probabilities (PIPs)** and **95% credible sets**,
both from individual-level data (`susie()`) and from summary statistics
(`susie_rss()`). `susie_rss()` combines GWAS $Z$-scores with an LD
matrix to undo the effect of LD within one locus. In the final session
(`GWAS11`), we apply the same idea to the whole genome: we combine
**GWAS summary statistics**, an **LD reference panel**
(`ldak --calc-cors`) and the per-SNP heritabilities from `GWAS8`
(`sumher_ldak.ind.hers`) to build **advanced polygenic scores**.

> **Other take-home messages**
>
> - **Loci often have several causal variants**: `SuSiE` finds them
>   without searching over all combinations, and gives each its own
>   credible set.
> - **Report PIPs and credible sets**: Strong LD shares the PIP between
>   SNPs, so the credible set is needed to capture the signal with at
>   least 95% probability.
> - **The LD matrix must match the GWAS**: For `susie_rss()`, use LD
>   from the GWAS sample or from a large reference panel of the same
>   ancestry, and check `estimate_s_rss()` for mismatch.